# Index Restoration Utility

This notebook restores the pre-configured Azure AI Search indexes (`hrdocs` and `healthdocs`) used throughout the LAB532 workshop. The indexes contain pre-processed documents with embeddings and semantic configurations, so you can focus on learning Knowledge Bases APIs rather than data preparation.

### Prerequisites

| Item | Details |
|------|---------|
| `.env` file | In the `notebooks/` directory. Must contain `BIGBUGS_ENDPOINT`, `BIGBUGS_API_KEY`, `AZURE_OPENAI_ENDPOINT`, and `AZURE_OPENAI_KEY`. |
| Python packages | `azure-search-documents`, `python-dotenv` (already in `requirements.txt`). |
| Data files | `../data/index-data/index.json`, `hrdocs-exported.jsonl`, `healthdocs-exported.jsonl` (checked in). |

> **Run this notebook once** before starting the Part 1–5 notebooks.

In [ ]:
import json
import os
import traceback

from azure.core.credentials import AzureKeyCredential
from azure.search.documents.aio import SearchClient
from azure.search.documents.indexes.aio import SearchIndexClient
from azure.search.documents.indexes.models import SearchIndex


async def restore_index(endpoint: str, index_name: str, index_file: str, records_file: str, azure_openai_endpoint: str):
    default_path = r"../data/index-data"
    log_message = print
    try:
        log_message(f"[{index_name}] Starting index restoration...")
        
        credential = AzureKeyCredential(os.environ["BIGBUGS_API_KEY"])
        async with SearchIndexClient(endpoint=endpoint, credential=credential) as client:
            index_file_path = os.path.join(default_path, index_file)
            log_message(f"[{index_name}] Reading index definition from: {index_file_path}")
            
            with open(index_file_path, "r", encoding="utf-8") as in_file:
                index_data = json.load(in_file)
                index = SearchIndex.deserialize(index_data)
                index.name = index_name
                index.vector_search.vectorizers[0].parameters.resource_url = azure_openai_endpoint
                index.vector_search.vectorizers[0].parameters.api_key = os.environ["AZURE_OPENAI_KEY"]
                log_message(f"[{index_name}] Creating/updating index in Azure AI Search...")
                await client.create_or_update_index(index)
                log_message(f"[{index_name}] Index created/updated successfully")

        async with SearchClient(endpoint=endpoint, index_name=index_name, credential=credential) as client:
            records_file_path = os.path.join(default_path, records_file)
            log_message(f"[{index_name}] Reading documents from: {records_file_path}")
            
            records = []
            total_uploaded = 0
            batch_count = 0
            
            with open(records_file_path, "r", encoding="utf-8") as in_file:
                for line_num, line in enumerate(in_file, 1):
                    try:
                        record = json.loads(line)
                        records.append(record)
                        
                        if len(records) >= 100:
                            batch_count += 1
                            log_message(f"[{index_name}] Uploading batch #{batch_count} ({len(records)} documents)...")
                            await client.upload_documents(documents=records)
                            total_uploaded += len(records)
                            records = []
                    except json.JSONDecodeError as e:
                        log_message(f"[{index_name}] WARNING: Skipping invalid JSON on line {line_num}: {e}")
                        continue

            if records:
                batch_count += 1
                log_message(f"[{index_name}] Uploading final batch #{batch_count} ({len(records)} documents)...")
                await client.upload_documents(documents=records)
                total_uploaded += len(records)
        
        log_message(f"[{index_name}] SUCCESS - Index restored! Total documents uploaded: {total_uploaded}")
        return True
        
    except FileNotFoundError as e:
        print(f"[{index_name}] ERROR - File not found: {e}")
        traceback.print_exc()
        return False
    except Exception as e:
        print(f"[{index_name}] ERROR - {type(e).__name__}: {e}")
        traceback.print_exc()
        return False


In [ ]:
from dotenv import load_dotenv

load_dotenv(override=True)

endpoint = os.environ["BIGBUGS_ENDPOINT"]
azure_openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]

print(f"Search endpoint : {endpoint}")
print(f"OpenAI endpoint : {azure_openai_endpoint}")

In [ ]:
# Restore hrdocs index
print("\n--- Processing hrdocs index ---")
await restore_index(
    endpoint, 
    "hrdocs", 
    "index.json", 
    "hrdocs-exported.jsonl", 
    azure_openai_endpoint
)

In [ ]:
# Restore healthdocs index
print("\n--- Processing healthdocs index ---")
await restore_index(
    endpoint, 
    "healthdocs", 
    "index.json", 
    "healthdocs-exported.jsonl", 
    azure_openai_endpoint 
)

## Next Steps

Once both indexes are restored successfully, you can proceed with the LAB532 notebooks.

➡️ Start with [Part 1: Standard KB with a Search Index knowledge source](part1-standard-foundry-iq-kb.ipynb)